# Getting started: LIBERO on HiPerGator

This notebook sets up LIBERO and the packages used by the existing dataset and skill notebooks. It downloads software and, optionally, the LIBERO-90 and LIBERO-10 demonstrations. Use a HiPerGator Jupyter session with network access and sufficient Blue storage. This is a starting environment, not a complete SmolVLA training environment.

## 1. Create a Python kernel

In a HiPerGator **terminal**, replace `FILLMEIN` with your username and run the following once. LIBERO documents Python 3.8.13. If your checkout is already present, skip the clone and place it at the shown path, or change the paths in the existing notebooks.

```bash
module load conda
mkdir -p /blue/uf-eig/FILLMEIN
cd /blue/uf-eig/FILLMEIN
git clone https://github.com/uf-eig/HSR_Paper_Replication.git HSR-Replication
conda create -p /blue/uf-eig/FILLMEIN/hsr-env python=3.8.13 pip -y
conda activate /blue/uf-eig/FILLMEIN/hsr-env
python -m pip install 'ipykernel<7'
python -m ipykernel install --user --name hsr-libero --display-name 'Python (HSR LIBERO)'
```

Reopen this notebook from the checkout, select **Python (HSR LIBERO)** in Jupyter, and run the cells below in order. The clone URL assumes the repository is available under `uf-eig`; use your actual checkout URL if needed. See [UF's Jupyter documentation](https://docs.rc.ufl.edu/interfaces/jupyter/).


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

project_root = Path('/blue/uf-eig/FILLMEIN/HSR-Replication')
libero_repo = project_root / 'LIBERO'
dataset_root = project_root / 'datasets'
print('Python:', sys.executable)
print('Project:', project_root)


Replace `FILLMEIN` in the cell above before running it. Check that the printed Python path points to `hsr-env` and that the project path exists. Package installation runs in the selected kernel's environment.

## 2. Clone and install LIBERO

The requirements file includes LIBERO's own pinned dependencies plus packages used by this project. The CUDA 11.3 PyTorch command follows the upstream LIBERO instructions; GPU availability depends on your allocation and compatible drivers. Downloads may take several minutes.


In [ ]:
if not project_root.joinpath('requirements.txt').is_file():
    raise FileNotFoundError('Update project_root to the HSR checkout containing requirements.txt.')
if not libero_repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Lifelong-Robot-Learning/LIBERO.git', str(libero_repo)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==1.11.0+cu113', 'torchvision==0.12.0+cu113', 'torchaudio==0.11.0', '--extra-index-url', 'https://download.pytorch.org/whl/cu113'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(libero_repo)], check=True)


Restart the kernel after installation, then run the path cell in section 1 again before continuing.

## 3. Configure paths and check LIBERO

A project-local LIBERO configuration avoids its first-run interactive prompt. The cache and dataset locations stay on Blue storage. `MUJOCO_GL=egl` requests headless GPU rendering; a GPU-backed session and working EGL are needed to render simulations.


In [ ]:
import yaml

cache_root = project_root.parent / 'cache'
os.environ['HF_HOME'] = str(cache_root)
os.environ['TORCH_HOME'] = str(cache_root)
os.environ['MUJOCO_GL'] = 'egl'
config_dir = project_root / '.libero-config'
config_dir.mkdir(parents=True, exist_ok=True)
os.environ['LIBERO_CONFIG_PATH'] = str(config_dir)
libero_package = libero_repo / 'libero' / 'libero'
config_path = config_dir / 'config.yaml'
if not config_path.exists():
    config_path.write_text(yaml.safe_dump({
        'benchmark_root': str(libero_package),
        'bddl_files': str(libero_package / 'bddl_files'),
        'init_states': str(libero_package / 'init_files'),
        'datasets': str(dataset_root),
        'assets': str(libero_package / 'assets'),
    }))

import torch
from libero.libero import benchmark

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Available suites:', list(benchmark.get_benchmark_dict()))


If CUDA is unavailable, confirm that your Jupyter session has a GPU and that the installed PyTorch build matches the cluster. Importing the benchmark does not test simulation rendering. For other notebooks, set `LIBERO_CONFIG_PATH` and `MUJOCO_GL` before importing LIBERO if you want to reuse this configuration.

## 4. Download demonstrations (optional)

The `libero_100` download contains the `libero_90` and `libero_10` directories used by the next notebook. This is a large download; set the switch to `True` when ready. Run it only where network access is available.


In [ ]:
download_datasets = False
if download_datasets:
    from huggingface_hub import snapshot_download

    dataset_root.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='yifengzhu-hf/LIBERO-datasets',
        repo_type='dataset',
        local_dir=str(dataset_root),
        allow_patterns=['libero_90/*.hdf5', 'libero_10/*.hdf5'],
    )

for suite in ['libero_90', 'libero_10']:
    print(suite, ':', len(list((dataset_root / suite).glob('*.hdf5'))), 'HDF5 files')


Once both directories contain demonstrations, continue with [01 — Dataset Exploration](01_dataset.ipynb), followed by [02 — Skill Discovery](02_skill_discovery.ipynb). The existing notebooks use the placeholder paths shown in the README; update `FILLMEIN` there as well.
